# Exploratory Data Analysis 

Analysis by Dustin Keate<br>
BSDA Program, WGU

## Data Description

https://www.kaggle.com/datasets/uradkr/s-and-p-500-analyst-rating-and-price-target-accuracy

164,231 sell-side analyst rating actions on S&P 500 stocks from 307 firms, December 2011 to June 2026. Every upgrade, downgrade, and initiation is matched against realized forward stock returns at 30, 90, 180, and 365 days — producing a directional accuracy flag per call. Analysts were right 48.8% of the time at 30 days, rising to 57.0% at one year.

## Initial Examination

Ingest ```.csv``` data and take an initial look at its size, structure, and completeness.

In [12]:
import pandas as pd
import numpy as np
import datetime as dt

df = pd.read_csv('../data/sp_500_analyst_rating_and_price_target_accuracy.csv')
df.sample(10)

,ticker,event_date,firm,from_grade,to_grade,action,price_target_action,current_price_target,prior_price_target,implied_direction,forward_return_30d_pct,forward_return_90d_pct,forward_return_180d_pct,forward_return_365d_pct,accuracy_30d,accuracy_90d,accuracy_180d,accuracy_365d
61412,FANG,2016-10-17 04:00:00,Stephens & Co.,Equal-Weight,Overweight,up,NaN,0.0,0.0,1,-2.866543,-0.029869,4.648176,2.130016,0.0,0.0,1.0,1.0
23756,BSX,2024-07-25 11:17:11,Baird,Outperform,Outperform,main,Raises,91.0,90.0,1,7.015889,17.761910,35.887424,42.930240,1.0,1.0,1.0,1.0
148348,UBER,2021-08-05 12:49:18,DA Davidson,NaN,Buy,main,Lowers,68.0,70.0,1,-4.922217,6.152778,-10.355234,-25.679130,0.0,1.0,0.0,0.0
36929,CPT,2023-04-19 11:49:41,Goldman Sachs,NaN,Neutral,main,Lowers,120.0,134.0,0,-0.383497,4.437053,-7.579786,-7.148423,NaN,NaN,NaN,NaN
19102,BAC,2016-11-08 08:57:17,Deutsche Bank,Buy,Hold,down,NaN,0.0,0.0,-1,35.500879,36.504563,41.895295,59.990352,0.0,0.0,0.0,0.0
421,AAPL,2025-10-31 21:07:13,Wells Fargo,Overweight,Overweight,main,Raises,300.0,290.0,1,4.809864,-4.379053,0.116487,NaN,1.0,0.0,1.0,NaN
109343,NOW,2021-01-28 18:34:25,Mizuho,NaN,Buy,main,Raises,610.0,600.0,1,0.402352,0.541280,5.060985,1.234126,1.0,1.0,1.0,1.0
101595,MRK,2012-12-20 17:29:48,Goldman Sachs,NaN,Neutral,main,Lowers,47.0,49.0,0,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN
159253,WM,2023-07-27 14:09:34,BMO Capital,Market Perform,Market Perform,main,Lowers,166.0,167.0,0,-3.677487,0.512088,14.135659,22.044958,NaN,NaN,NaN,NaN
158559,WELL,2015-03-16 13:00:00,Citigroup,NaN,Neutral,main,Raises,83.0,69.0,0,0.305812,-9.164252,-13.735082,-8.073233,NaN,NaN,NaN,NaN


In [13]:
df.shape

(164231, 18)

In [14]:
df.dtypes

ticker                         str
event_date                     str
firm                           str
from_grade                     str
to_grade                       str
action                         str
price_target_action            str
current_price_target       float64
prior_price_target         float64
implied_direction            int64
forward_return_30d_pct     float64
forward_return_90d_pct     float64
forward_return_180d_pct    float64
forward_return_365d_pct    float64
accuracy_30d               float64
accuracy_90d               float64
accuracy_180d              float64
accuracy_365d              float64
dtype: object

In [15]:
df.count()

ticker                     164231
event_date                 164231
firm                       164222
from_grade                  94124
to_grade                   164096
action                     164231
price_target_action        151448
current_price_target       164231
prior_price_target         164231
implied_direction          164231
forward_return_30d_pct     162815
forward_return_90d_pct     158629
forward_return_180d_pct    153242
forward_return_365d_pct    142009
accuracy_30d               116592
accuracy_90d               113769
accuracy_180d              110064
accuracy_365d              102310
dtype: int64

In [16]:
df.nunique()

ticker                        502
event_date                 152609
firm                          306
from_grade                     46
to_grade                       46
action                          5
price_target_action             8
current_price_target         2210
prior_price_target           2185
implied_direction               3
forward_return_30d_pct     100826
forward_return_90d_pct      99076
forward_return_180d_pct     96950
forward_return_365d_pct     92259
accuracy_30d                    2
accuracy_90d                    2
accuracy_180d                   2
accuracy_365d                   2
dtype: int64

Initial Observations:

1. There are ```164231``` entries with 18 different columns consisting of entry metadata, analyst predictions (from and to), ```forward_return``` percentage, and ```accuracy``` assessments.
2. ```event_date``` column is a ```str``` object. The entries will be more valuable as a ```datetime``` object if date becomes relevant.
3. 9 rows are missing the ```firm``` entry.
4. ```forward_return``` percentage drops off naturally as the prediction ```event_date``` is closer to the end date of the data set. Data cannot exist without time travel.
5. It appears an abnormal number of rows are missing ```accuracy``` entries. This warrants additional investigation.


In [17]:
df[df['firm'].isna()]

,ticker,event_date,firm,from_grade,to_grade,action,price_target_action,current_price_target,prior_price_target,implied_direction,forward_return_30d_pct,forward_return_90d_pct,forward_return_180d_pct,forward_return_365d_pct,accuracy_30d,accuracy_90d,accuracy_180d,accuracy_365d
6190,AEP,2014-11-06 10:53:04,NaN,Outperform,Market Perform,down,NaN,0.0,0.0,-1,0.000000,1.749783,-7.802993,-8.394161,0.0,0.0,1.0,1.0
27686,CCL,2016-11-22 23:30:53,NaN,Market Perform,Market Outperform,up,NaN,0.0,0.0,1,1.236219,7.958269,20.612376,32.328741,1.0,1.0,1.0,1.0
45738,DG,2016-10-10 13:06:15,NaN,NaN,Hold,main,Lowers,0.0,0.0,0,3.011064,6.569461,0.919371,18.244725,NaN,NaN,NaN,NaN
70052,GOOG,2012-07-17 13:33:00,NaN,NaN,Outperform,main,Lowers,730.0,750.0,1,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0
105166,MU,2015-07-28 17:43:00,NaN,NaN,Buy,up,NaN,0.0,0.0,1,-20.658216,-16.050641,-46.936703,-28.151899,0.0,0.0,0.0,0.0
105561,NCLH,2016-11-22 23:31:58,NaN,Outperform,Market Perform,down,NaN,0.0,0.0,-1,7.776400,19.751555,25.490679,37.416152,0.0,0.0,0.0,0.0
111034,NTRS,2016-10-10 13:06:14,NaN,NaN,Hold,main,Raises,0.0,0.0,0,7.048716,24.113923,19.741670,31.783742,NaN,NaN,NaN,NaN
132928,SNPS,2016-10-10 13:08:22,NaN,NaN,Buy,main,Raises,0.0,0.0,1,-2.542934,-0.792602,17.800520,36.525757,0.0,0.0,1.0,1.0
138842,TEL,2014-01-22 13:00:00,NaN,Hold,Buy,up,Raises,75.0,55.0,1,0.000000,0.000000,0.000000,-0.585740,0.0,0.0,0.0,0.0


Rows that are missing ```firm``` entries are not abnormal except for the identified missing entry. Since the ```firm``` column is a critical element of my hypothesis, remove those rows.

In [18]:
df[df['accuracy_30d'].isna()].sample(10)

,ticker,event_date,firm,from_grade,to_grade,action,price_target_action,current_price_target,prior_price_target,implied_direction,forward_return_30d_pct,forward_return_90d_pct,forward_return_180d_pct,forward_return_365d_pct,accuracy_30d,accuracy_90d,accuracy_180d,accuracy_365d
123707,PSA,2025-04-09 11:30:04,Mizuho,NaN,Neutral,init,Announces,287.0,0.0,0,7.715477,4.061782,5.137203,8.287267,NaN,NaN,NaN,NaN
116923,PAYX,2025-10-01 12:47:54,BMO Capital,Market Perform,Market Perform,main,Lowers,140.0,143.0,0,-5.628579,-7.680073,-23.395125,NaN,NaN,NaN,NaN,NaN
151242,UPS,2022-09-22 11:02:51,Barclays,NaN,Equal-Weight,main,Lowers,180.0,200.0,0,-0.184694,6.238952,13.307437,-4.966774,NaN,NaN,NaN,NaN
46090,DGX,2012-11-05 12:54:00,Jefferies,NaN,Hold,main,Lowers,59.0,63.0,0,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN
131575,SHW,2019-10-02 12:34:19,UBS,NaN,Neutral,main,Raises,555.0,475.0,0,7.140332,8.068252,-10.539638,28.824010,NaN,NaN,NaN,NaN
71573,GPN,2016-07-29 17:01:00,Nomura,NaN,NaN,main,Raises,66.0,64.0,0,1.701242,-2.920346,4.877229,26.468057,NaN,NaN,NaN,NaN
62965,FE,2020-08-04 11:22:12,Mizuho,NaN,Neutral,main,Lowers,28.5,45.0,0,0.511030,2.357782,8.083200,34.857910,NaN,NaN,NaN,NaN
74732,HII,2024-11-01 17:41:13,Deutsche Bank,Hold,Hold,main,Lowers,191.0,273.0,0,3.666602,4.017760,23.511480,72.147485,NaN,NaN,NaN,NaN
53998,EIX,2025-10-28 14:09:01,Wells Fargo,NaN,Equal-Weight,init,Announces,56.0,0.0,0,5.066898,11.592439,25.637416,NaN,NaN,NaN,NaN,NaN
63045,FE,2016-05-09 08:01:09,Barclays,NaN,Equal-Weight,main,Raises,36.0,33.0,0,1.728761,-0.956411,4.814979,-10.216014,NaN,NaN,NaN,NaN


According to this sample, all entries that do not have ```accuracy``` entries are either:
- Missing associated ```forward_return``` entries for reasons already discussed, OR
- Have a ```to_grade``` entry that is associated with a neutral outlook, such as "Equal-Weight," "Hold," or "Neutral."

Both of these are normal and explainable.

**A new observation**: Many rows that have ```event_dates``` prior to approximately 1 Jan 2015 have ```forward_return``` percentages of ```0.0```.

## Initial Data Cleaning

Before performing a more complex analysis, cleaning steps identified above will be completed.

1. Convert ```event_date``` from a ```str``` object to a ```datetime``` object.
2. Remove entries with a missing ```firm``` entry.

In [19]:
df['event_date'] = pd.to_datetime(df['event_date'], format='%Y-%m-%d %H:%M:%S')

df.dropna(subset=['firm'], inplace=True)

## Further Examination of Cleaned Data

In [20]:
df[(df['event_date'] < dt.datetime(2014, 12, 5)) & (df['event_date'] > dt.datetime(2014, 12, 3))][['event_date', 'forward_return_30d_pct', 'accuracy_30d']].sample(10)

,event_date,forward_return_30d_pct,accuracy_30d
140646,2014-12-03 05:00:00,0.000000,0.0
20810,2014-12-04 13:19:15,-0.543874,0.0
16807,2014-12-04 12:55:25,-1.598547,NaN
16804,2014-12-04 14:00:00,-1.598547,NaN
142237,2014-12-04 12:16:30,-7.351048,NaN
15443,2014-12-04 14:00:00,1.202769,NaN
72857,2014-12-03 05:00:00,0.000000,0.0
84675,2014-12-04 14:00:00,-0.698406,NaN
16805,2014-12-04 13:36:58,-1.598547,0.0
119282,2014-12-04 09:26:47,-0.475448,0.0


In [21]:
df[df['event_date'] < dt.datetime(2014, 12, 4)].shape

(17723, 18)

Rows with an ```event_date``` on or before ```2014-12-03``` are missing critical ```forward_return``` data that will likely be relevant when performing further analysis.

The original dataset has ```164231``` rows. The missing data affects ```17723``` rows. Removing this will represent a data loss of under 11%.

The dataset contains ratings from December 2011 to June 2026. Of this, December 2011 to November 2014 are missing data. This represents a loss of about 36 months of the 175 months represented, about 20%.

## Further Data Cleaning

Remove rows missing accurate ```forward_return``` entries.

In [22]:
df=df[df['event_date'] > dt.datetime(2014, 12, 4)]
df.shape

(146499, 18)

## Data Exploration

